This notebook is used to synthesize the trained models with HLS4MLs backend Vitis Unified. Some synthesis include bitfile-generation, and varies with strategy.

Note: This failed to synthesize in the block-level diagram because of hung-/zombie-processes. Though it would never fit on the KV260 anyway, so there is no real value in the final leg of the synthesis.

In [1]:
import os
import sys
import time
import numpy as np
import matplotlib.pyplot as plt
import keras
import tensorflow as tf
from sklearn.metrics import accuracy_score

# Load Vitis into path
os.environ['PATH'] = os.environ['XILINX_VITIS'] + '/bin:' + os.environ['PATH']

I0000 00:00:1778783792.283643    2720 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: SSE4.1 SSE4.2, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
model_to_test = 'jettag-baseline'

model_configs = [
    {
        "description": "Baseline acc=0.7685 Trying without softmax",
        "model_revision": "baseline",
        "keras_model_path": "jet_baseline/the_baseline/model_the_baseline_acc=0.7677.keras",
        "hls4ml_strategy": "latency",
        "hls4ml_generate_bitfile": True,
        "hls4ml_revision": "acc=0.7677_withoutSoftmax_VU_DA_bitfile",
    },
#    {
#        "description": "Baseline acc=0.7685 latency",
#        "model_revision": "baseline",
#        "keras_model_path": "jet_baseline/the_baseline/model_the_baseline_acc=0.7677.keras",
#        "hls4ml_strategy": "latency",
#        "hls4ml_generate_bitfile": True,
#        "hls4ml_revision": "acc=0.7685_VU_latency_bitfile",
#    },
]

In [3]:
# Load dataset which is preprocessed in another notebook
#X_train_val = np.load('Data/x_train_val.npy')
x_test = np.load('Data/x_test.npy')
#y_train_val = np.load('Data/y_train_val.npy')
#y_test = np.load('Data/y_test.npy')
#classes = np.load('Data/classes.npy', allow_pickle=True)


In [4]:
import os

def prepare_directory(model_config):
    output_dir = os.path.join(
        os.path.dirname(os.path.abspath(model_config["keras_model_path"])),
        f"hls4ml_prj_{model_config['hls4ml_revision']}",
    )
    os.makedirs(output_dir, exist_ok=True)

    description = f"""
    Description of HLS4ML-project.

    {model_config['description']}

    - Bitfile: {model_config['hls4ml_generate_bitfile']}
    - Environment: devenv-vu-hgq+da (environment-HGQ+DA.yml)
    - Target Device: KV260 (xck26-sfvc784-2LV-c)
    - Dataset: HLS4ML LHC Jets
    - Vivado/Vitis: 2025.2
    - Model Architecture: {model_to_test}
    - Model Revision: {model_config['model_revision']}
    - HLS4ML Revision: {model_config['hls4ml_revision']}

    The model summary is in the parent-directory, `summary.txt`
    """
    with open(os.path.join(output_dir, "description.md"), "w", encoding="utf-8") as f:
        f.write(description)

    return output_dir

In [5]:
from keras.models import load_model
import hgq.layers
import hls4ml

def compile_model(keras_model_path, output_dir, hls4ml_strategy):
    model = load_model(keras_model_path)
    
    hls_config = hls4ml.utils.config_from_keras_model(model, granularity='name')
    
    strategy = 'Distributed Arithmetic' if hls4ml_strategy == 'DA' else hls4ml_strategy
    hls_config['Model']['Strategy'] = strategy # https://fastmachinelearning.org/hls4ml/api/configuration.html#top-level-configuration

    hls_model = hls4ml.converters.convert_from_keras_model( 
        model,    
        backend     =   'vitisunified',
        hls_config  =   hls_config,
        output_dir  =   output_dir, 
        board       =   'kv260',
        part        =   'xck26-sfvc784-2LV-c',
        clock_period=   '5',
        strategy    =   strategy
    )
    hls_model.compile()
    return hls_model

In [6]:
# Hotfix for crashing, see README
os.environ['LD_PRELOAD'] = '/lib/x86_64-linux-gnu/libudev.so.1'

process_model_durations = []

# Run through every model
for i, model_config in enumerate(model_configs):
    print(f"Processing model {i+1}/{len(model_configs)}: {model_config['description']}")
    print(f"%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%")
    start_time = time.time()
    output_dir = prepare_directory(model_config)
    
    hls_model = compile_model(
        model_config["keras_model_path"],
        output_dir,
        model_config["hls4ml_strategy"],
    )

    # Create complete bitfile (Vitis Unified-backend) or IP-block (Vitis-backend)
    hls_model.build(
        synth=True,
        bitfile=model_config["hls4ml_generate_bitfile"],
        csim=False # Simulation (CSIM and COSIM) needs input_data_tb and output_data_tb https://fastmachinelearning.org/hls4ml/autodoc/hls4ml.converters.html#hls4ml.converters.convert_from_keras_model
    )
    
    elapsed_time = time.time() - start_time
    process_model_durations.append({
        'model': model_config['description'],
        'time_seconds': elapsed_time,
        'time_minutes': elapsed_time / 60
    })
    print(f"\nModel {i+1} completed in {elapsed_time:.2f} seconds ({elapsed_time/60:.2f} minutes)")


Processing model 1/1: Baseline acc=0.7685 Trying without softmax
%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

****** v++ v2025.2 (64-bit)
  **** SW Build 6295257 on 2025-11-13-01:29:13
  **** Start of session at: Thu May 14 18:36:37 2026
    ** Copyright 1986-2022 Xilinx, Inc. All Rights Reserved.
    ** Copyright 2022-2025 Advanced Micro Devices, Inc. All Rights Reserved.

  **** HLS Build v2025.2 6295257
INFO: [HLS 200-2005] Using work_dir /work/development/jettag/jet_baseline/the_baseline/hls4ml_prj_acc=0.7677_withoutSoftmax_VU_DA_bitfile/vitis_workspace/myproject/vitis_unified_project 
INFO: [HLS 200-2176] Writing Vitis IDE component file /work/development/jettag/jet_baseline/the_baseline/hls4ml_prj_acc=0.7677_withoutSoftmax_VU_DA_bitfile/vitis_workspace/myproject/vitis_unified_project/vitis-comp.json
INFO: [HLS 200-10] Creating and opening component '/work/development/jettag/jet_baseline/the_baseline/hls4ml_prj_acc=0.7677_withoutSoftmax_VU_DA_bitfile/vitis_workspace/myproject/vitis_

ERROR: The following input file does not exist: myproject.xclbin


In [7]:
import pandas as pd

timestamp = time.strftime("%Y%m%d_%H%M%S")
timing_df = pd.DataFrame(process_model_durations)
timing_df = timing_df[["model", "time_seconds", "time_minutes"]]
timing_df.to_csv(f"{model_to_test}/timing_summary_{timestamp}.csv", index=False)

display(timing_df)

total_time = timing_df["time_seconds"].sum()
average_time = timing_df["time_seconds"].mean()
print(f"\nTotal time: {total_time:.2f} seconds ({total_time/60:.2f} minutes)")
print(f"Average time per model: {average_time:.2f} seconds ({average_time/60:.2f} minutes)")

OSError: Cannot save file into a non-existent directory: 'jettag-baseline'

Copy bitfile and hwh from export-directory to directory for onboard-verification.
Make sure the driver and testdata is copied manually.

In [ ]:
import glob
import shutil

target_dir = '../../onboard-verification/jettag/DUT'

for model_config in model_configs:
    export_dir = os.path.join(
        os.path.dirname(os.path.abspath(model_config["keras_model_path"])),
        f"hls4ml_prj_{model_config['hls4ml_revision']}",
        'export'
    )
    model_target_dir = os.path.join(
        target_dir,
        f"{model_config['model_revision']}_{model_config['hls4ml_revision']}"
    )
    os.makedirs(model_target_dir, exist_ok=True)

    for extension in ('bit', 'hwh'):
        matches = sorted(glob.glob(os.path.join(export_dir, f'*.{extension}')))
        if not matches:
            print(f"ERROR: No .{extension} file found in {export_dir}")
            continue
        shutil.copy2(matches[0], model_target_dir)
        print(f"Copied {matches[0]} -> {model_target_dir}")

Copied /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/jettag/jettag-hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7117_ebops=303_VU_DA_bitfile/export/system.bit -> ../../onboard-verification/jettag/DUT/Training_AdaptiveHP_acc=0.7117_ebops=303_VU_DA_bitfile
Copied /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/jettag/jettag-hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7117_ebops=303_VU_DA_bitfile/export/system.hwh -> ../../onboard-verification/jettag/DUT/Training_AdaptiveHP_acc=0.7117_ebops=303_VU_DA_bitfile
Copied /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/jettag/jettag-hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7117_ebops=303_VU_latency_bitfile/export/system.bit -> ../../onboard-verification/jettag/DUT/Training_AdaptiveHP_acc=0.7117_ebops=303_VU_latency_bitfile
Copied /home/ncgadmin/Bachelor/HLS4ML_testbench_KV260/development/jettag/jettag-hgq2/Training_AdaptiveHP/hls4ml_prj_acc=0.7117_ebops=303_VU_latency_bitfile/export/system.hwh -> ../../onboard-veri

adhoc needed the baseline

In [ ]:
# Save baseline predictions of 
from keras.models import load_model

model = load_model("jet_baseline/the_baseline/model_the_baseline_acc=0.7685.keras")
y_baseline = model.predict(x_test)
np.save("Data/y_baseline.npy",y_baseline)

I0000 00:00:1778174000.190315  407335 service.cc:153] XLA service 0x78fdb40340c0 initialized for platform Host (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778174000.190357  407335 service.cc:161]   StreamExecutor [0]: Host, Default Version (Driver: 0.0.0; Runtime: 0.0.0; Toolkit: 0.0.0; DNN: 0.0.0)
I0000 00:00:1778174000.225131  407335 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.


 287/5188 ━━━━━━━━━━━━━━━━━━━━ 2s 526us/step

I0000 00:00:1778174000.507416  407335 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


5188/5188 ━━━━━━━━━━━━━━━━━━━━ 3s 530us/step
